ARTI308 - Machine Learning

# Project Notebook 8: Model Comparison
## Heart Disease Prediction

Each model notebook (04 to 07) saved its results in `outputs/metrics_*.csv`.
Here we put them together and answer the main question: **which model should we choose, and do the ensemble models really beat the baselines?**

Run notebooks 04, 05, 06 and 07 first.

## 1. Load the saved results

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

files = sorted(glob.glob("../outputs/metrics_*.csv"))
print("Found result files:", [f.split("/")[-1] for f in files])

results = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

expected = {"Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"}
missing = expected - set(results["model"])
if missing:
    print("WARNING - missing results for:", missing, "(run their notebooks first)")

results.round(3)

## 2. Compare the metrics

In [ ]:
cols = ["accuracy", "precision", "recall", "f1", "cv_accuracy", "cv_recall"]
summary = results.set_index("model")[["type"] + cols].round(3)
summary.sort_values("recall", ascending=False)

**Observation:**

*(Write what you notice after running the cell above.)*

In [ ]:
plot_df = results.melt(id_vars=["model", "type"], value_vars=["accuracy", "precision", "recall", "f1"],
                       var_name="metric", value_name="score")

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model")
plt.title("Test-set metrics by model")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 3. The cost of mistakes
In a medical problem, a **false negative** (a sick patient predicted healthy) is more dangerous than a false positive.

In [ ]:
errors = results.set_index("model")[["TP", "TN", "FP", "FN"]]
errors

In [ ]:
errors[["FP", "FN"]].plot(kind="bar", figsize=(8, 4))
plt.title("False positives vs false negatives")
plt.ylabel("Number of patients (test set)")
plt.xticks(rotation=0)
plt.show()

**Observation:**

*(Write what you notice after running the cell above.)*

## 4. Baseline vs. ensemble
Compare the best baseline with the best ensemble.

In [ ]:
by_type = results.groupby("type")[["accuracy", "recall", "f1", "cv_accuracy", "cv_recall"]].agg(["mean", "max"]).round(3)
by_type

**Observation:**

*(Write what you notice after running the cell above.)*

## 5. Final decision (write this yourself)

- Which model do you choose?
- Why (accuracy, **recall**, false negatives, interpretability)?
- Do the ensemble models justify their extra complexity compared with the baselines?
- What are the limitations (small dataset of ~900 patients, single train/test split, etc.)?

> Note: the test set is small, so small differences between models may be due to chance. The cross-validation columns give a more stable picture.